# SurgiVision — compare trained models
Loads existing **best checkpoints**. No training or dataset copying occurs.

Run cells in order. In Colab, choose a GPU runtime, then mount Drive below. Locally, skip the Drive mount and set ROOT to your project directory. Update the three checkpoint paths if your models have moved.

Outputs: overall and per-class CSVs, comparison chart, confusion matrices, PR curves, and a record of checkpoint hashes/settings. Scores use the same prepared validation set (not an independent test benchmark).

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import gc

import torch
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ultralytics import YOLO


# --------------------------------------------------
# 1. Paths — change ROOT if your project is elsewhere
# --------------------------------------------------
ROOT = Path.home() / "Desktop" /"AppliedAI" / "SurgiVision-AI"

DATASET = (
    ROOT / "prepared_datasets" / "20261103T011448501787Z"
)

MODEL_PATHS = {
    "YOLOv8n": ROOT / "models" / "v8nbest.pt",
    "YOLOv8m": ROOT / "models" / "v8best.pt",
    "YOLOv9c": ROOT / "models" / "v9cbest.pt",
}

NAMES = [
    "Scalpel",
    "Straight Dissection Clamp",
    "Straight Mayo Scissor",
    "Curved Mayo Scissor",
]

SPLIT = "val"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
IMGSZ = 640
BATCH = 16


# --------------------------------------------------
# 2. Validate model and dataset paths
# --------------------------------------------------
for name, path in MODEL_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name} checkpoint not found:\n{path}")

source_config = DATASET / "data.yaml"
if not source_config.is_file():
    raise FileNotFoundError(source_config)

config = yaml.safe_load(source_config.read_text())
if not isinstance(config, dict) or "names" not in config:
    raise ValueError("data.yaml must contain class names.")

configured_names = config["names"]
if isinstance(configured_names, dict):
    configured_names = [
        value
        for key, value in sorted(
            configured_names.items(),
            key=lambda item: int(item[0]),
        )
    ]

if configured_names != NAMES:
    raise ValueError(
        f"Dataset class names/order differ:\n{configured_names}"
    )

image_dir = DATASET / "images" / SPLIT
label_dir = DATASET / "labels" / SPLIT

for folder in (image_dir, label_dir):
    if not folder.is_dir():
        raise FileNotFoundError(folder)

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
images = sorted(
    path
    for path in image_dir.iterdir()
    if path.is_file() and path.suffix.lower() in extensions
)

if not images:
    raise ValueError(f"No evaluation images found in {image_dir}")

missing_labels = [
    str(label_dir / f"{image.stem}.txt")
    for image in images
    if not (label_dir / f"{image.stem}.txt").is_file()
]

if missing_labels:
    raise FileNotFoundError(
        "Missing labels:\n" + "\n".join(missing_labels[:10])
    )


# --------------------------------------------------
# 3. Create a fresh comparison folder
# --------------------------------------------------
comparison_id = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)
OUT = ROOT / "comparisons" / comparison_id
OUT.mkdir(parents=True, exist_ok=False)

# Save a separate config; preserve the original.
config["path"] = str(DATASET.resolve())
config[SPLIT] = f"images/{SPLIT}"

EVAL_CONFIG = OUT / "evaluation_data.yaml"
EVAL_CONFIG.write_text(
    yaml.safe_dump(config, sort_keys=False)
)

print(f"Device: {DEVICE}")
print(f"Validation images: {len(images)}")
print(f"Output folder: {OUT}")


# --------------------------------------------------
# 4. Evaluate both best checkpoints identically
# --------------------------------------------------
overall_rows = []
class_rows = []

for name, checkpoint in MODEL_PATHS.items():
    print(f"\nEvaluating {name}...")

    model = YOLO(str(checkpoint))

    model_names = model.names
    if isinstance(model_names, dict):
        model_names = [
            value
            for key, value in sorted(
                model_names.items(),
                key=lambda item: int(item[0]),
            )
        ]
    else:
        model_names = list(model_names)

    if model_names != NAMES:
        raise ValueError(
            f"{name} checkpoint class names/order differ:\n{model_names}"
        )

    metrics = model.val(
        data=str(EVAL_CONFIG),
        split=SPLIT,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        conf=0.001,
        iou=0.7,
        half=False,
        augment=False,
        seed=0,
        workers=0,
        plots=True,
        project=str(OUT),
        name=name,
        verbose=False,
    )

    overall_rows.append({
        "Model": name,
        "Precision": float(metrics.box.mp),
        "Recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
        "Parameters": sum(
            parameter.numel()
            for parameter in model.model.parameters()
        ),
        "Weights_MB": checkpoint.stat().st_size / (1024 ** 2),
    })

    for index, class_id in enumerate(metrics.box.ap_class_index):
        precision, recall, ap50, ap5095 = (
            metrics.box.class_result(index)
        )

        class_rows.append({
            "Model": name,
            "Class": NAMES[int(class_id)],
            "Precision": float(precision),
            "Recall": float(recall),
            "AP50": float(ap50),
            "AP50-95": float(ap5095),
        })

    # Save after each model in case a later evaluation fails.
    pd.DataFrame(overall_rows).to_csv(
        OUT / "model_comparison.csv",
        index=False,
    )
    pd.DataFrame(class_rows).to_csv(
        OUT / "per_class_comparison.csv",
        index=False,
    )

    del metrics, model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# --------------------------------------------------
# 5. Display overall results
# --------------------------------------------------
summary = pd.DataFrame(overall_rows).set_index("Model")
per_class = pd.DataFrame(class_rows)

score_columns = [
    "Precision",
    "Recall",
    "mAP50",
    "mAP50-95",
]

print("\nOverall validation results:")
display(
    summary.style.format({
        **{column: "{:.2%}" for column in score_columns},
        "Parameters": "{:,.0f}",
        "Weights_MB": "{:.2f}",
    })
)

difference = (
    summary.loc["YOLOv8m", score_columns]
    - summary.loc["YOLOv8n", score_columns]
) * 100

print("\nYOLOv8m minus YOLOv8n — percentage points:")
display(
    difference.rename("Difference (pp)")
    .to_frame()
    .style.format("{:+.2f}")
)


# --------------------------------------------------
# 6. Save overall comparison chart
# --------------------------------------------------
ax = (summary[score_columns] * 100).plot.bar(
    figsize=(11, 6),
    ylim=(0, 105),
    rot=0,
)

ax.set_title("YOLOv8n vs YOLOv8m — validation results")
ax.set_ylabel("Score (%)")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=8, padding=3)

plt.tight_layout()
plt.savefig(
    OUT / "model_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


# --------------------------------------------------
# 7. Display and plot per-class localization scores
# --------------------------------------------------
class_comparison = per_class.pivot(
    index="Class",
    columns="Model",
    values="AP50-95",
).reindex(NAMES)

print("\nPer-class AP@0.5:0.95:")
display(class_comparison.style.format("{:.2%}"))

ax = (class_comparison * 100).plot.barh(
    figsize=(11, 6),
    xlim=(0, 105),
)

ax.set_title("Per-class AP@0.5:0.95")
ax.set_xlabel("Score (%)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.savefig(
    OUT / "per_class_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print(f"\nComparison complete. Results saved to:\n{OUT}")

Device: cpu
Validation images: 784
Output folder: /Users/aathiabhishek/Desktop/AppliedAI/SurgiVision-AI/comparisons/20261111T212449281355Z

Evaluating YOLOv8n...
Ultralytics 8.3.167 🚀 Python-3.12.7 torch-2.6.0 CPU (Apple M2 Pro)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 180.1±56.1 MB/s, size: 54.0 KB)


val: Scanning /Users/aathiabhishek/Desktop/AppliedAI/SurgiVision-AI/prepared_dat

val: New cache created: /Users/aathiabhishek/Desktop/AppliedAI/SurgiVision-AI/prepared_datasets/20261103T011448501787Z/labels/val.cache



                 Class     Images  Instances      Box(P          R      mAP50  m


                   all        784       1087      0.961       0.96      0.989      0.928
Speed: 0.4ms preprocess, 131.4ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /Users/aathiabhishek/Desktop/AppliedAI/SurgiVision-AI/comparisons/20261111T212449281355Z/YOLOv8n

Evaluating YOLOv8m...
Ultralytics 8.3.167 🚀 Python-3.12.7 torch-2.6.0 CPU (Apple M2 Pro)
Model summary (fused): 92 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 540.4±718.0 MB/s, size: 66.5 KB)


val: Scanning /Users/aathiabhishek/Desktop/AppliedAI/SurgiVision-AI/prepared_dat
                 Class     Images  Instances      Box(P          R      mAP50  m


... (7 outputs left out)


Per-class AP@0.5:0.95:


Model,YOLOv8m,YOLOv8n,YOLOv9c
Class,,,
Scalpel,92.83%,89.59%,92.77%
Straight Dissection Clamp,92.12%,87.78%,91.40%
Straight Mayo Scissor,98.00%,96.66%,98.01%
Curved Mayo Scissor,97.80%,97.13%,97.76%


<Figure size 1100x600 with 1 Axes>


Comparison complete. Results saved to:
/Users/aathiabhishek/Desktop/AppliedAI/SurgiVision-AI/comparisons/20261111T212449281355Z
